# T3.7 Colab 확인: 정식 트레이너의 GPU 결정성·재개 후 일치·3시간 상한

`src/sr_pipeline/engine/trainer.py`의 `train_run`을 **Colab T4 GPU**(`device="cuda"`)에서 실행해, 로컬 CPU 테스트로는 확인할 수 없는 항목을 봅니다. 로직은 `src/`에 있고 이 노트북은 실행·확인만 합니다.

- 셀 1~2: 저장소(YongMin 브랜치) 가져오기, 최신 코드·GPU 확인
- 셀 3: 점검용 합성 패치 풀과 실행 조건 준비
- 셀 4 **확인 ①** GPU 결정성: 학습하는 구성 9개(baseline, A, B, C, Bp, AB, AC, BC, ABC)를 같은 시드로 2회 학습 → 에폭 손실·가중치가 비트 단위로 같은가
- 셀 5 **확인 ②** 재개 후 일치: 2에폭 저장 뒤 런타임이 끊긴 것처럼 멈추고 재개 → 한 번에 끝까지 돈 결과와 같은가
- 셀 6 **확인 ③** 상한: 상한에 걸리면 `incomplete`로 기록하고, 재개해도 더 학습하지 않는가
- 셀 7: 요약 (**이 출력을 그대로 복사해 전달**)
- 셀 8 (참고, ①이 FAIL일 때만): 어떤 연산이 비결정적인지 진단
- 셀 9 (참고, 통과 기준 아님): 파일럿 규모 1에폭 학습 시간 → 3시간에 몇 에폭이 들어가는지 가늠

**런타임 유형: T4 GPU.** 토큰·Drive는 필요 없습니다(저장소는 Public, 데이터는 합성). 셀 1~7은 수 분, 셀 9는 수 분이 더 걸립니다.

합성 패치 풀과 학습 설정 값은 이 노트북 안에서만 쓰는 **점검용 값**이며 공정 비교 조건이 아닙니다(`pilot_determined`는 T4.2에서 확정). `configs/`는 읽기만 합니다.

①②③이 모두 OK면 T3.7 통과 조건의 "Colab T4 확인"을 충족합니다. ①이 FAIL이면 코드를 고치기 전에 셀 8 출력과 함께 보고합니다(결정적 알고리즘 강제 여부는 팀이 정합니다).

In [1]:
import os
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull --ff-only          # 이미 받아 둔 경우 최신으로
else:
    !git clone --branch YongMin --depth 1 {REPO_URL} /content/repo
!git -C /content/repo log --oneline -3             # "fix: T3.7 교차 리뷰 반영 ..." 커밋이 보여야 함. 안 보이면 로컬에서 git push 먼저
assert os.path.isdir("/content/repo/src/sr_pipeline"), "clone 실패: REPO_URL/푸시 확인"
print("셀 1 OK")

remote: Enumerating objects: 681, done.
remote: Counting objects: 100% (681/681), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 590 (delta 356), reused 512 (delta 309), pack-reused 0 (from 0)
Receiving objects: 100% (590/590), 7.37 MiB | 27.97 MiB/s, done.
Resolving deltas: 100% (356/356), completed with 53 local objects.
From https://github.com/minyi-k03/VideoProcessing-Project
   5d9bff5..909246c  YongMin    -> origin/YongMin
Updating 5d9bff5..909246c
Fast-forward
 docs/03-plan.md                          |   5 +-
 docs/SPEC.md                             |   6 +-
 notebooks/t2_5_pilot_colab.ipynb         |  47 ++-
 scripts/generate_pilot_bicubic.py        |  24 +-
 scripts/run_pilot_diagnosis.py           |  15 +-
 src/sr_pipeline/data/dataset.py          |   5 +
 src/sr_pipeline/diagnosis/diagnostics.py |   6 +
 src/sr_pipeline/diagnosis/provenance.py  |   1 +
 src/sr_pipeline/engine/pilot.py          |   8 +
 src/sr_pipeline/engine/trainer.py        | 334 

In [2]:
import sys, importlib, subprocess, platform
sys.path.insert(0, "/content/repo/src")
sys.path_importer_cache.pop("/content/repo/src", None)   # 클론 전에 import를 시도했던 커널의 "경로 없음" 캐시 제거
importlib.invalidate_caches()
# 이전 실행에서 import한 옛 모듈이 커널 메모리에 남아 있으면 git pull 후에도 옛 코드가 실행된다 -> 캐시를 비운다.
for name in [m for m in sys.modules if m == "sr_pipeline" or m.startswith("sr_pipeline.")]:
    del sys.modules[name]
import numpy as np, torch
from sr_pipeline.engine import trainer
from sr_pipeline.data.dataset import PatchPoolDataset
assert hasattr(trainer, "DIRTY_SUFFIX") and hasattr(PatchPoolDataset, "hr_patch_size"), \
    "trainer.py가 옛 버전입니다: 로컬에서 git push 한 뒤 셀 1을 다시 실행하세요."

assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > T4 GPU"
DEVICE = "cuda"
COMMIT = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(),
      "|", torch.cuda.get_device_name(0))
print("commit", COMMIT)
print("셀 2 OK")

torch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800 | Tesla T4
commit 909246c86b90407b350babb67a18a3b78acb5dff
셀 2 OK


In [3]:
# 점검용 합성 패치 풀과 실행 조건. 학습 설정 값은 이 노트북에서만 쓰는 점검용 값이다(공정 비교 조건 아님).
# 실제 configs/fair_comparison.yaml을 읽어 pilot_determined만 점검용 값으로 채운 사본을 /content에 쓰고,
# resolve_run_config로 실행 조건을 확정한다(구성·시드 검증은 실제 코드 그대로).
import copy, json, shutil, time
from pathlib import Path
import yaml
from sr_pipeline.config.schema import resolve_run_config

REPO = Path("/content/repo")
ROOT = Path("/content/t3_7_check")
shutil.rmtree(ROOT, ignore_errors=True)   # 이 노트북의 산출물만 지운다(다시 실행해도 같은 조건에서 시작)

SCALE, PATCH_HR, POOL_SIZE, POOL_ID = 4, 96, 512, "colab_check_synthetic_pool"
DATA_DIR = ROOT / "data"
pool_dir = DATA_DIR / "patch_pool"
pool_dir.mkdir(parents=True)
pool_rng = np.random.default_rng(2024)
np.save(pool_dir / "hr.npy", pool_rng.integers(0, 256, (POOL_SIZE, PATCH_HR, PATCH_HR, 3), dtype=np.uint8))
np.save(pool_dir / "lr.npy", pool_rng.integers(0, 256, (POOL_SIZE, PATCH_HR // SCALE, PATCH_HR // SCALE, 3), dtype=np.uint8))
(pool_dir / "metadata.json").write_text(json.dumps({"patch_pool_id": POOL_ID, "scale": SCALE}), encoding="utf-8")

FAIR = yaml.safe_load((REPO / "configs" / "fair_comparison.yaml").read_text(encoding="utf-8"))
CHECK_PILOT = {
    "patch_pool_id": POOL_ID, "patch_pool_size": POOL_SIZE, "patch_size_hr": PATCH_HR,
    "patches_per_epoch": 256, "epochs": 3, "batch_size": 16, "learning_rate": 1e-3,
    "optimizer": "adam", "scheduler": "none", "augmentation": "rot90x4_hflip",
    "edge_loss_lambda": 0.1, "edge_top_q": 10,
    "representative_images": {name: {"median": "-", "worst": "-"} for name in FAIR["fixed_rules"]["datasets"]},
}
TRAINABLE = ["baseline", "A", "B", "C", "Bp", "AB", "AC", "BC", "ABC"]

def make_config(config_id, seed=0, max_hours=None, **pilot):
    fair = copy.deepcopy(FAIR)
    fair["pilot_determined"] = {**copy.deepcopy(CHECK_PILOT), **pilot}
    if max_hours is not None:
        fair["fixed_rules"]["max_hours"] = max_hours
    path = ROOT / "fair_comparison_check.yaml"
    path.write_text(yaml.safe_dump(fair, allow_unicode=True), encoding="utf-8")
    return resolve_run_config(path, REPO / "configs" / "experiments.yaml", config_id, seed)

def train(label, run_config, **kwargs):
    # 실행마다 <ROOT>/<label>/{records,checkpoints}에 쓴다.
    return trainer.train_run(run_config, DATA_DIR, ROOT / label / "records", ROOT / label / "checkpoints",
                             commit=COMMIT, device=DEVICE, **kwargs)

def weights_of(result):
    return {name: tensor.detach().cpu().clone() for name, tensor in result.model.state_dict().items()}

def compare(a, b):
    # (에폭 손실 목록이 같고 가중치가 비트 단위로 같은가, 가중치 최대 절대 차이)
    wa, wb = weights_of(a), weights_of(b)
    bitwise = wa.keys() == wb.keys() and all(torch.equal(wa[k], wb[k]) for k in wa)
    diff = max((wa[k].double() - wb[k].double()).abs().max().item() for k in wa)
    return a.epoch_losses == b.epoch_losses and bitwise, diff

for config_id in TRAINABLE:
    assert make_config(config_id).trainable, config_id
results = {}
print("셀 3 OK: 합성 풀", (POOL_SIZE, PATCH_HR, PATCH_HR, 3), "| 구성", TRAINABLE)

셀 3 OK: 합성 풀 (512, 96, 96, 3) | 구성 ['baseline', 'A', 'B', 'C', 'Bp', 'AB', 'AC', 'BC', 'ABC']


In [4]:
# 확인 ①: GPU에서 같은 구성·시드를 두 번 학습하면 에폭 손실과 가중치가 비트 단위로 같은가 (US32-33)
reference = {}
for config_id in TRAINABLE:
    cfg = make_config(config_id)
    first = train(f"{config_id}_first", cfg)
    second = train(f"{config_id}_second", cfg)
    ok, diff = compare(first, second)
    reference[config_id] = first
    results[f"① 결정성 {config_id}"] = ok
    print(f"{'OK  ' if ok else 'FAIL'} {config_id:8s} 같은 시드 2회 동일 | 가중치 최대 차이 {diff:.3e} | 에폭 손실 "
          + ", ".join(f"{loss:.6f}" for loss in first.epoch_losses))

other_seed = train("baseline_seed1", make_config("baseline", seed=1))
results["① 다른 시드 -> 다른 결과 (baseline 0 vs 1)"] = not compare(reference["baseline"], other_seed)[0]
print("OK  " if results["① 다른 시드 -> 다른 결과 (baseline 0 vs 1)"] else "FAIL", "다른 시드 -> 다른 결과")

OK   baseline 같은 시드 2회 동일 | 가중치 최대 차이 0.000e+00 | 에폭 손실 0.310085, 0.264691, 0.259612
FAIL A        같은 시드 2회 동일 | 가중치 최대 차이 2.811e-06 | 에폭 손실 0.474717, 0.429275, 0.423898
OK   B        같은 시드 2회 동일 | 가중치 최대 차이 0.000e+00 | 에폭 손실 0.264683, 0.188141, 0.181417
OK   C        같은 시드 2회 동일 | 가중치 최대 차이 0.000e+00 | 에폭 손실 0.291103, 0.261187, 0.256320
OK   Bp       같은 시드 2회 동일 | 가중치 최대 차이 0.000e+00 | 에폭 손실 0.248243, 0.182965, 0.174494
FAIL AB       같은 시드 2회 동일 | 가중치 최대 차이 1.299e-06 | 에폭 손실 0.375026, 0.298671, 0.291695
FAIL AC       같은 시드 2회 동일 | 가중치 최대 차이 6.847e-05 | 에폭 손실 0.479313, 0.430015, 0.422291
OK   BC       같은 시드 2회 동일 | 가중치 최대 차이 0.000e+00 | 에폭 손실 0.188680, 0.168282, 0.165712
FAIL ABC      같은 시드 2회 동일 | 가중치 최대 차이 2.636e-05 | 에폭 손실 0.313032, 0.279866, 0.276093
OK   다른 시드 -> 다른 결과


In [5]:
# 확인 ②: 런타임이 끊긴 뒤 재개한 결과가 한 번에 끝까지 돈 결과(확인 ①의 첫 실행)와 같은가 (통과 조건 "재개 후 결과 동일", US36)
# ①이 FAIL인 구성은 ②도 FAIL일 수 있다. 그때 원인은 재개가 아니라 GPU 비결정성이다(셀 8로 진단).
class CutClock:
    # 실제 시계를 쓰다가 fail_at_call번째 호출에서 런타임이 끊긴 것처럼 예외를 던진다.
    # 트레이너는 에폭마다 시작·끝에 시계를 한 번씩 읽으므로 5번째 호출 = 2에폭을 저장한 뒤 3번째 에폭 시작 순간.
    def __init__(self, fail_at_call):
        self.fail_at_call, self.calls = fail_at_call, 0
    def __call__(self):
        self.calls += 1
        if self.calls == self.fail_at_call:
            raise ConnectionError("런타임 끊김(모의)")
        return time.perf_counter()

for config_id in TRAINABLE:
    cfg = make_config(config_id)
    label = f"{config_id}_cut"
    try:
        train(label, cfg, clock=CutClock(fail_at_call=5))
        raise AssertionError("끊김 모의가 일어나지 않았습니다")
    except ConnectionError:
        pass
    run_id = trainer.build_run_id(COMMIT, config_id, 0)
    state = torch.load(ROOT / label / "checkpoints" / run_id / trainer.STATE_FILENAME, map_location="cpu", weights_only=True)
    saved_epochs = len(state["epoch_losses"])
    resumed = train(label, cfg, resume=True)
    same, diff = compare(resumed, reference[config_id])
    ok = same and saved_epochs == 2 and resumed.status == "complete"
    results[f"② 재개 후 일치 {config_id}"] = ok
    print(f"{'OK  ' if ok else 'FAIL'} {config_id:8s} 끊기기 전 저장 {saved_epochs}에폭 -> 재개 후 {resumed.epochs_done}에폭 "
          f"| 한 번에 돈 결과와 가중치 최대 차이 {diff:.3e}")

OK   baseline 끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 0.000e+00
FAIL A        끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 2.980e-08
OK   B        끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 0.000e+00
OK   C        끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 0.000e+00
OK   Bp       끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 0.000e+00
FAIL AB       끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 1.300e-06
FAIL AC       끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 1.160e-04
OK   BC       끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 0.000e+00
FAIL ABC      끊기기 전 저장 2에폭 -> 재개 후 3에폭 | 한 번에 돈 결과와 가중치 최대 차이 3.470e-05


In [6]:
# 확인 ③: 3시간 상한에 걸리면 incomplete로 기록하고, 재개해도 더 학습하지 않는가 (US16, 20)
# 상한을 1e-9시간(약 3.6마이크로초)으로 두면 첫 에폭을 끝낸 뒤 두 번째 에폭을 시작하기 전에 멈춘다.
capped_cfg = make_config("baseline", max_hours=1e-9)
capped = train("capped", capped_cfg)
capped_log = json.loads((ROOT / "capped" / "records" / capped.run_id / trainer.TRAIN_LOG_FILENAME).read_text(encoding="utf-8"))
results["③ 상한 -> incomplete (1/3에폭, train_log.json 포함)"] = (
    (capped.status, capped.epochs_done, capped.epochs_total) == ("incomplete", 1, 3)
    and (capped_log["status"], capped_log["epochs_done"], capped_log["epochs_total"]) == ("incomplete", 1, 3)
    and capped_log["train_minutes"] > 0)
again = train("capped", capped_cfg, resume=True)
results["③ 상한에 걸린 실행 재개 -> 추가 학습 없이 incomplete"] = (
    again.status == "incomplete" and again.epochs_done == 1 and compare(again, capped)[0])
print("상한 실행:", capped.status, f"{capped.epochs_done}/{capped.epochs_total}에폭", f"train_minutes={capped.train_minutes:.6f}")
print("재개:", again.status, f"{again.epochs_done}/{again.epochs_total}에폭")

상한 실행: incomplete 1/3에폭 train_minutes=0.001490
재개: incomplete 1/3에폭


In [7]:
# 이 출력을 그대로 복사해 전달한다(T3.7 통과 조건 ③ Colab T4 확인 근거).
for name, ok in results.items():
    print("OK  " if ok else "FAIL", name)
print("전체:", "PASS" if all(results.values()) else "FAIL")
print("commit  :", COMMIT)
print("device  :", torch.cuda.get_device_name(0))
print("versions: torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(),
      "| numpy", np.__version__)
print("platform:", platform.platform())

OK   ① 결정성 baseline
FAIL ① 결정성 A
OK   ① 결정성 B
OK   ① 결정성 C
OK   ① 결정성 Bp
FAIL ① 결정성 AB
FAIL ① 결정성 AC
OK   ① 결정성 BC
FAIL ① 결정성 ABC
OK   ① 다른 시드 -> 다른 결과 (baseline 0 vs 1)
OK   ② 재개 후 일치 baseline
FAIL ② 재개 후 일치 A
OK   ② 재개 후 일치 B
OK   ② 재개 후 일치 C
OK   ② 재개 후 일치 Bp
FAIL ② 재개 후 일치 AB
FAIL ② 재개 후 일치 AC
OK   ② 재개 후 일치 BC
FAIL ② 재개 후 일치 ABC
OK   ③ 상한 -> incomplete (1/3에폭, train_log.json 포함)
OK   ③ 상한에 걸린 실행 재개 -> 추가 학습 없이 incomplete
전체: FAIL
commit  : 909246c86b90407b350babb67a18a3b78acb5dff
device  : Tesla T4
versions: torch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800 | numpy 2.4.6
platform: Linux-6.6.159+-x86_64-with-glibc2.39


In [8]:
# 참고(①이 FAIL일 때만): 결정적 알고리즘을 강제(warn_only)하고 다시 2회 학습해, 비결정 연산 이름과
# 강제했을 때 같아지는지 본다. 코드를 바꿀지(예: torch.use_deterministic_algorithms, CUBLAS_WORKSPACE_CONFIG)는
# 이 출력을 보고 팀이 정한다. 이 셀은 프로세스 전역 설정을 끝에 원래대로 되돌린다.
import warnings
failed = [c for c in TRAINABLE if not results[f"① 결정성 {c}"]]
if not failed:
    print("①이 모두 OK라 진단할 것이 없습니다.")
else:
    torch.use_deterministic_algorithms(True, warn_only=True)
    try:
        for config_id in failed:
            cfg = make_config(config_id)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                a = train(f"{config_id}_det_a", cfg)
                b = train(f"{config_id}_det_b", cfg)
            ops = sorted({str(w.message).split(" does not have a deterministic")[0].strip()
                          for w in caught if "determinis" in str(w.message)})
            same, diff = compare(a, b)
            print(f"{config_id:8s} 결정적 알고리즘 강제 시 2회 동일: {same} (가중치 최대 차이 {diff:.3e})")
            for op in ops:
                print("    경고:", op[:200])
    finally:
        torch.use_deterministic_algorithms(False)

A        결정적 알고리즘 강제 시 2회 동일: True (가중치 최대 차이 0.000e+00)
AB       결정적 알고리즘 강제 시 2회 동일: True (가중치 최대 차이 0.000e+00)
AC       결정적 알고리즘 강제 시 2회 동일: True (가중치 최대 차이 0.000e+00)
ABC      결정적 알고리즘 강제 시 2회 동일: True (가중치 최대 차이 0.000e+00)


In [9]:
# 참고(통과 기준 아님): 파일럿 규모(에폭당 10,000패치, 배치 16, HR 패치 96)로 구성별 1에폭 학습 시간을 잰다.
# 3시간 상한 안에 몇 에폭이 들어가는지 가늠하는 용도다(T4.1·T4.2 판단 근거). 합성 데이터라 손실 값은 의미 없다.
# train_minutes는 데이터 증강·배치 구성을 포함한 에폭 학습 시간의 합이다(체크포인트 저장 시간은 제외).
for config_id in TRAINABLE:
    timed = train(f"{config_id}_timing", make_config(config_id, patches_per_epoch=10_000, epochs=1))
    print(f"{config_id:8s} 1에폭 {timed.train_minutes * 60:6.1f}초 -> 3시간 상한에 약 {180 / timed.train_minutes:5.0f}에폭")

baseline 1에폭    2.9초 -> 3시간 상한에 약  3764에폭
A        1에폭    4.0초 -> 3시간 상한에 약  2712에폭
B        1에폭    3.2초 -> 3시간 상한에 약  3413에폭
C        1에폭    3.1초 -> 3시간 상한에 약  3539에폭
Bp       1에폭    3.3초 -> 3시간 상한에 약  3296에폭
AB       1에폭    4.2초 -> 3시간 상한에 약  2590에폭
AC       1에폭    3.4초 -> 3시간 상한에 약  3155에폭
BC       1에폭    3.2초 -> 3시간 상한에 약  3352에폭
ABC      1에폭    4.4초 -> 3시간 상한에 약  2429에폭
